# Deliverable 2 — Reality: The Disclosure Delay

**The "drop."** The STOCK Act says politicians must file their trades — it doesn't say they must file fast (30 days from learning, 45 from the transaction, $200 fine). Backtest **v2** re-enters every trade on the day a retail follower could actually act: **filing date + 1**.

- Same trades, same 60-day hold as Deliverable 1 — only the entry date changes
- Headline stat: the **average transaction → disclosure delay** across the full dataset

Outputs → `output/2-disclosure-delay/`: `v1_vs_v2_equity.mp4`, `trade_timeline.mp4`, `viral_trades_reality.mp4`, `trade_results_v2.csv`, `summary_stats.json`

In [1]:
import sys
import json
import warnings
from pathlib import Path
from datetime import timedelta
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as mpl_animation
from matplotlib.ticker import FuncFormatter

REPO_ROOT = Path.cwd().resolve().parent
load_dotenv(REPO_ROOT / '.env')
sys.path.insert(0, str(REPO_ROOT))

from lib import brand, animation, data_quiver, data_massive, backtest
from lib.brand import BG, GREEN, OFF_WHITE, RED, OLIVE, GRID, VCR
from audio_engine import Cue, render_track, ticks_every

warnings.filterwarnings('ignore')
brand.apply_theme()
print('Imports OK')

Imports OK


In [2]:
HOLD_DAYS = 60
PRICE_END = '2026-07-02'

OUT_DIR = Path('../output/2-disclosure-delay')
OUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR = Path('cache/audio')
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {OUT_DIR.resolve()}')

Output directory: G:\congress-trades-investigation\output\2-disclosure-delay


In [ ]:
# Same trades as Deliverable 1 (fresh pull, identical cleaning)
raw_congress = data_quiver.fetch_congress_trades()
congress_df = data_quiver.clean_congress_buys(raw_congress)
print(f'{len(congress_df):,} buy trades | {congress_df["Ticker"].nunique()} tickers')

In [ ]:
# ── The headline stat: how long until the public actually finds out? ─────────
# Computed on ALL disclosed transactions (buys and sells) with valid dates.
all_tx = raw_congress.dropna(subset=['TransactionDate', 'ReportDate'])
delay_all = (all_tx['ReportDate'] - all_tx['TransactionDate']).dt.days
delay_all = delay_all[delay_all >= 0]  # a handful of bad filings report before trading

delay_stats = {
    'avg_delay_days': float(delay_all.mean()),
    'median_delay_days': float(delay_all.median()),
    'pct_over_45_days': float((delay_all > 45).mean()),
    'max_delay_days': int(delay_all.max()),
    'n_transactions': int(len(delay_all)),
}
print(f"Average transaction → disclosure delay: {delay_stats['avg_delay_days']:.1f} days")
print(f"Median: {delay_stats['median_delay_days']:.0f} days "
      f"| filed later than the 45-day limit: {delay_stats['pct_over_45_days']:.1%} "
      f"| worst: {delay_stats['max_delay_days']:,} days")

Average transaction → disclosure delay: 47.9 days
Median: 28 days | filed later than the 45-day limit: 13.0% | worst: 4,112 days


In [ ]:
# Shared price cache (already populated by Deliverable 1 — cache hits only)
fetch_start = (congress_df['TransactionDate'].min() - timedelta(days=5)).strftime('%Y-%m-%d')
tickers = ['SPY'] + sorted(congress_df['Ticker'].unique().tolist())
price_cache = data_massive.load_price_cache(tickers, fetch_start, PRICE_END)
spy_prices = price_cache['SPY']

Price cache: 3778 tickers cached | 0 to fetch
Done: 3520 tickers with data | 241 empty (delisted/bad symbol)


In [ ]:
# ── Both engines on identical trades ──────────────────────────────────────────
print('v1 — perfect info (transaction-date entry):')
v1_results = backtest.run_backtest(congress_df, price_cache, spy_prices,
                                   hold_days=HOLD_DAYS, entry='transaction')
print('\nv2 — reality (filing date + 1 entry):')
v2_results = backtest.run_backtest(congress_df, price_cache, spy_prices,
                                   hold_days=HOLD_DAYS, entry='disclosure')

v1_curve = backtest.equity_curve(v1_results, price_cache, spy_prices)
v2_curve = backtest.equity_curve(v2_results, price_cache, spy_prices)

v1 — perfect info (transaction-date entry):


44,270 trades complete | 3403 skipped
Avg return: 1.93% | SPY: 2.32% | Excess: -0.39%
Win rate: 56.04% | Beat SPY: 47.24%

v2 — reality (filing date + 1 entry):


43,749 trades complete | 3924 skipped
Avg return: 2.05% | SPY: 2.32% | Excess: -0.27%
Win rate: 56.10% | Beat SPY: 46.83%


In [ ]:
# ── Save data + headline numbers ──────────────────────────────────────────────
summary = {
    'engine_v1': backtest.summarize(v1_results, v1_curve),
    'engine_v2': backtest.summarize(v2_results, v2_curve),
    'disclosure_delay': delay_stats,
    'hold_days': HOLD_DAYS,
    'v2_entry_rule': 'filing date + 1 day',
}
summary['v1_minus_v2_avg_return'] = (summary['engine_v1']['avg_trade_return']
                                     - summary['engine_v2']['avg_trade_return'])

v2_results.to_csv(OUT_DIR / 'trade_results_v2.csv', index=False)
v1_curve.to_csv(OUT_DIR / 'equity_curve_v1.csv', index=False)
v2_curve.to_csv(OUT_DIR / 'equity_curve_v2.csv', index=False)
with open(OUT_DIR / 'summary_stats.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"v1 avg excess vs SPY: {summary['engine_v1']['avg_excess_return']:+.2%}")
print(f"v2 avg excess vs SPY: {summary['engine_v2']['avg_excess_return']:+.2%}")
print(f"Edge lost to the disclosure delay: {summary['v1_minus_v2_avg_return']:+.2%} per trade")

v1 avg excess vs SPY: -0.39%
v2 avg excess vs SPY: -0.27%
Edge lost to the disclosure delay: -0.12% per trade


In [ ]:
# ── Visual 1: side-by-side equity curves — the dream vs the drop ─────────────
# Both panels share a y-scale so the collapse is unmissable. v2 drawn in red
# (comparison color per STYLE.md); SPY off-white in both.
def animate_v1_v2(v1c, v2c, output_path, draw_s=12.0, hold_s=2.5, audio_wav=None):
    n_draw = int(draw_s * brand.VIDEO_FPS)
    n_total = n_draw + int(hold_s * brand.VIDEO_FPS)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(brand.FIG_W, brand.FIG_H), facecolor=BG)
    fig.suptitle('SAME TRADES - DIFFERENT ENTRY DAY', color=OFF_WHITE,
                 fontsize=22, fontfamily=VCR, fontweight='bold', y=0.97)

    y_lo = min(v1c['CumPortfolio'].min(), v1c['CumSPY'].min(),
               v2c['CumPortfolio'].min(), v2c['CumSPY'].min()) * 0.95
    y_hi = max(v1c['CumPortfolio'].max(), v1c['CumSPY'].max(),
               v2c['CumPortfolio'].max(), v2c['CumSPY'].max()) * 1.08

    panels = []
    for ax, curve, color, subtitle in [
        (ax1, v1c, GREEN, 'v1 - BUY THE DAY THEY BOUGHT'),
        (ax2, v2c, RED,   'v2 - BUY WHEN YOU COULD (FILED + 1)'),
    ]:
        ax.set_facecolor(BG)
        ax.set_title(subtitle, color=color, fontsize=13, pad=12, fontfamily=VCR)
        ax.grid(True, color=GRID, linewidth=0.6, alpha=0.6, zorder=0)
        ax.set_axisbelow(True)
        for s in ax.spines.values():
            s.set_edgecolor(GRID)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        dates = curve['Date'].values
        ax.set_xlim(dates[0], dates[-1])
        ax.set_ylim(y_lo, y_hi)
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
        ax.tick_params(labelsize=9)
        ax.axhline(backtest.INITIAL_CAPITAL, color=GRID, linewidth=0.8,
                   linestyle='--', alpha=0.7)
        ln_spy, = ax.plot([], [], color=OFF_WHITE, linewidth=2, alpha=0.85, label='SPY')
        ln_strat, = ax.plot([], [], color=color, linewidth=2.5, label='Strategy')
        ax.legend(facecolor=BG, edgecolor=GRID, labelcolor=OFF_WHITE,
                  loc='upper left', prop={'family': VCR, 'size': 11})
        val_txt = ax.text(0.97, 0.95, '', transform=ax.transAxes, color=color,
                          fontsize=14, ha='right', va='top', fontfamily=VCR)
        idx = np.linspace(0, len(curve) - 1, n_draw, dtype=int)
        panels.append((curve, ln_strat, ln_spy, val_txt, idx))

    def update(frame):
        f = min(frame, n_draw - 1)
        artists = []
        for curve, ln_strat, ln_spy, val_txt, idx in panels:
            i = idx[f]
            d = curve['Date'].values[:i + 1]
            ln_strat.set_data(d, curve['CumPortfolio'].values[:i + 1])
            ln_spy.set_data(d, curve['CumSPY'].values[:i + 1])
            val_txt.set_text(f"${curve['CumPortfolio'].iloc[i]:,.0f}")
            artists += [ln_strat, ln_spy, val_txt]
        return artists

    ani = mpl_animation.FuncAnimation(fig, update, frames=n_total,
                                      interval=1000 / brand.VIDEO_FPS, blit=True)
    print(f'Rendering {output_path.name}  ({n_total} frames)...')
    animation.save_animation(ani, fig, output_path, audio_wav)


DRAW_S, HOLD_S = 12.0, 2.5
wav = render_track(
    ticks_every('tick', 1.0, DRAW_S - 1.0, 2.0, gain_db=-8)
    + [Cue('error_buzz', DRAW_S - 0.6, gain_db=-6),      # reality lands
       Cue('resolve_tone', DRAW_S + 0.8, gain_db=-6)],
    duration_s=DRAW_S + HOLD_S,
    out_path=AUDIO_DIR / 'd2_sidebyside.wav',
)
animate_v1_v2(v1_curve, v2_curve, OUT_DIR / 'v1_vs_v2_equity.mp4',
              draw_s=DRAW_S, hold_s=HOLD_S, audio_wav=wav)

findfont: Failed to find font weight bold, now using 400.


Rendering v1_vs_v2_equity.mp4  (870 frames)...


Saved v1_vs_v2_equity.mp4  (1.0 MB)


In [ ]:
# ── Pick the timeline trade: a big v1 winner where the delay actually cost you ─
# Requirements: a real filing delay (>=20 days, so the gap reads on screen) and a
# big v1→v2 return gap (the delay must be the story). Among qualifying trades,
# prefer household names (Pelosi first — she's the script's headliner).
key = ['Representative', 'Ticker', 'TransactionDate']
merged = v1_results.merge(v2_results, on=key, suffixes=('_v1', '_v2'))
merged['Gap'] = merged['TradeReturn_v1'] - merged['TradeReturn_v2']
merged['DelayDays'] = (merged['ReportDate_v1'] - merged['TransactionDate']).dt.days

HIGH_PROFILE = ['Pelosi', 'Greene', 'Tuberville', 'Crenshaw', 'Gottheimer', 'Khanna']
eligible = merged.nlargest(300, 'TradeReturn_v1')
eligible = eligible[(eligible['DelayDays'] >= 20) & (eligible['Gap'] >= 0.25)]

pick = None
for name in HIGH_PROFILE:
    hits = eligible[eligible['Representative'].str.contains(name, case=False)]
    if not hits.empty:
        pick = hits.nlargest(1, 'Gap').iloc[0]
        break
if pick is None:
    pick = eligible.nlargest(1, 'Gap').iloc[0]

print(f"Timeline trade: {pick['Representative']} — {pick['Ticker']}")
print(f"  bought {pick['TransactionDate'].date()} | filed {pick['ReportDate_v1'].date()} "
      f"({pick['DelayDays']:.0f} days later)")
print(f"  v1 return: {pick['TradeReturn_v1']:+.1%} | v2 return: {pick['TradeReturn_v2']:+.1%}")

Timeline trade: Josh Gottheimer — EBS
  bought 2020-06-15 | filed 2020-07-09 (24 days later)
  v1 return: +87.7% | v2 return: +10.6%


In [ ]:
# ── Visual 2: single-trade timeline — purchase → filing → retail entry → result ─
# Price line draws first, then the four events land one at a time and breathe.
# Labels are vertically staggered: FILED and YOU COULD ACT are 1 day apart.
def animate_trade_timeline(pick, prices, output_path, audio_wav=None,
                           draw_s=6.0, event_gap_s=2.5, hold_s=2.5):
    t_buy = pick['TransactionDate']
    t_filed = pick['ReportDate_v1']
    t_retail = pick['EntryDate_v2']
    t_exit_v2 = pick['ExitDate_v2']

    window = prices[(prices.index >= t_buy - timedelta(days=10)) &
                    (prices.index <= t_exit_v2 + timedelta(days=10))]
    dates, px = window.index.values, window.values

    events = [
        (t_buy,    f"BOUGHT {t_buy:%b %d}",  GREEN,     (8, 72)),
        (t_filed,  f"FILED {t_filed:%b %d}  (+{(t_filed - t_buy).days}d)", OFF_WHITE, (8, 46)),
        (t_retail, 'YOU COULD ACT', RED,     (8, 20)),
        (t_exit_v2, f"v1 {pick['TradeReturn_v1']:+.0%}  vs  v2 {pick['TradeReturn_v2']:+.0%}", RED, (-8, 36)),
    ]

    fps = brand.VIDEO_FPS
    n_draw = int(draw_s * fps)
    event_frames = [n_draw + int(i * event_gap_s * fps) for i in range(len(events))]
    n_total = event_frames[-1] + int((event_gap_s + hold_s) * fps)

    last = str(pick['Representative']).split()[-1].upper()
    fig, ax = animation.new_axes(
        f"{last} - {pick['Ticker']}: THE DELAY IS THE EDGE", fontsize=19)
    ax.set_xlim(dates[0], dates[-1])
    pad = (px.max() - px.min()) * 0.18
    ax.set_ylim(px.min() - pad, px.max() + pad * 1.6)
    ax.set_ylabel('Price ($)', color=OFF_WHITE, fontsize=13, fontfamily=VCR)
    ax.tick_params(labelsize=11)

    line, = ax.plot([], [], color=GREEN, linewidth=2.2, alpha=0.9, zorder=2)
    idx = np.linspace(0, len(px) - 1, n_draw, dtype=int)

    marks, texts = [], []
    for when, label, color, offset in events:
        pos = min(window.index.searchsorted(when), len(window) - 1)
        y = float(window.iloc[pos])
        vline = ax.axvline(when, color=color, linewidth=1.4, linestyle='--',
                           alpha=0.0, zorder=3)
        dot, = ax.plot([when], [y], 'o', color=color, markersize=9, alpha=0.0, zorder=4)
        txt = ax.annotate(label, xy=(when, y), xytext=offset,
                          textcoords='offset points', color=color, fontsize=13,
                          fontfamily=VCR, alpha=0.0, zorder=5,
                          ha='right' if offset[0] < 0 else 'left')
        marks.append((vline, dot))
        texts.append(txt)

    def update(frame):
        f = min(frame, n_draw - 1)
        i = idx[f]
        line.set_data(dates[:i + 1], px[:i + 1])
        artists = [line]
        for k, ef in enumerate(event_frames):
            a = animation.smooth_step((frame - ef) / (0.4 * fps))
            vline, dot = marks[k]
            vline.set_alpha(0.75 * a)
            dot.set_alpha(a)
            texts[k].set_alpha(a)
            artists += [vline, dot, texts[k]]
        return artists

    ani = mpl_animation.FuncAnimation(fig, update, frames=n_total,
                                      interval=1000 / fps, blit=True)
    print(f'Rendering {output_path.name}  ({n_total} frames)...')
    animation.save_animation(ani, fig, output_path, audio_wav)
    return n_total / fps


DRAW_S, EVENT_GAP_S, HOLD_S = 6.0, 2.5, 2.5
total_s = DRAW_S + 3 * EVENT_GAP_S + EVENT_GAP_S + HOLD_S
wav = render_track(
    [Cue('keystroke', DRAW_S + i * EVENT_GAP_S, gain_db=-4) for i in range(4)]
    + [Cue('data_blip', DRAW_S + 0.15, gain_db=-6),
       Cue('error_buzz', DRAW_S + 3 * EVENT_GAP_S + 0.4, gain_db=-6)],
    duration_s=total_s,
    out_path=AUDIO_DIR / 'd2_timeline.wav',
)
animate_trade_timeline(pick, price_cache[pick['Ticker']],
                       OUT_DIR / 'trade_timeline.mp4', audio_wav=wav,
                       draw_s=DRAW_S, event_gap_s=EVENT_GAP_S, hold_s=HOLD_S)

findfont: Failed to find font weight bold, now using 400.


Rendering trade_timeline.mp4  (1110 frames)...


Saved trade_timeline.mp4  (0.6 MB)


18.5

In [ ]:
# ── Visual 3: the "viral" v1 winners, re-run under v2 ─────────────────────────
# The trades that would go viral on social media — and what a follower actually got.
viral = merged.nlargest(8, 'TradeReturn_v1').iloc[::-1]  # best at top of chart

def viral_label(row):
    last = str(row['Representative']).split()[-1]
    return f"{row['Ticker']} - {last}"

GROW_S, GAP_S, HOLD_S = 1.5, 1.5, 3.0
total_s = GROW_S + GAP_S + GROW_S + HOLD_S
wav = render_track(
    [Cue('bar_grow', 0.1), Cue('data_blip', GROW_S, gain_db=-4),
     Cue('bar_grow', GROW_S + GAP_S, gain_db=-3),
     Cue('error_buzz', GROW_S + GAP_S + GROW_S, gain_db=-6)],
    duration_s=total_s,
    out_path=AUDIO_DIR / 'd2_viral.wav',
)

animation.animate_paired_bars(
    labels=[viral_label(r) for _, r in viral.iterrows()],
    values_a=viral['TradeReturn_v1'].tolist(),
    values_b=viral['TradeReturn_v2'].tolist(),
    title='THE "VIRAL" TRADES - WHAT YOU ACTUALLY GOT',
    output_path=OUT_DIR / 'viral_trades_reality.mp4',
    label_a='bought their day', label_b='bought when disclosed',
    grow_seconds=GROW_S, gap_seconds=GAP_S, hold_seconds=HOLD_S,
    audio_wav=wav,
)

viral.to_csv(OUT_DIR / 'viral_trades_v1_vs_v2.csv', index=False)
print(f"Avg v1 return of the viral 8: {viral['TradeReturn_v1'].mean():+.1%}")
print(f"Avg v2 return of the same 8: {viral['TradeReturn_v2'].mean():+.1%}")

findfont: Failed to find font weight bold, now using 400.


Rendering viral_trades_reality.mp4  (450 frames @ 60 fps)...


Saved viral_trades_reality.mp4  (0.2 MB)
Avg v1 return of the viral 8: +443.6%
Avg v2 return of the same 8: +350.5%
